# الفصل 23: خط إنتاج ما بعد التدريب

> طبقة الحدس بعد كل الجهد المبذول في الفصول 18-22 (بيانات نظيفة، Tokenizer مُحسَّن، حجم مثالي وفق Chinchilla، تدريب موزَّع مستقر)، ينتج لدينا ما يُسمَّى النموذج الأساسي (Base Model).

- **الجزء:** 5 — بناء LLM من الصفر
- **الفصل:** 23
- **المستوى:** خبير
- **زمن القراءة:** 25 دقيقة
- **المتطلبات:** الفصل 22: حفظ نقاط التفتيش ومراقبة التدريب
- **الوسوم:** SFT، Reward Model، RLHF، DPO

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch23-post-training.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-05-building-llms%2Fch23-post-training.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-05-building-llms/ch23-post-training.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch23-post-training.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch23-post-training.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch23-post-training.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** torch
- **GPU:** يستخدم هذا الدفتر PyTorch أو نماذج كبيرة؛ يُفضّل GPU. يمكنك البدء بنماذج صغيرة على CPU، لكن التدريب الكامل سيكون بطيئًا.
- **خلايا ثقيلة:** قد تستغرق خلايا التدريب/التحميل وقتًا طويلًا. شغّلها خلية بخية، وقلّل `batch_size` أو استخدم نموذجًا أصغر عند الحاجة.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("torch", "torch")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch23-post-training.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-05-building-llms%2Fch23-post-training.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-05-building-llms/ch23-post-training.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch23-post-training.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch23-post-training.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch23-post-training.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> طبقة الحدس بعد كل الجهد المبذول في الفصول 18-22 (بيانات نظيفة، Tokenizer مُحسَّن، حجم مثالي وفق Chinchilla، تدريب موزَّع مستقر)، ينتج لدينا ما يُسمَّى النموذج الأساسي (Base Model).

> **بيانات الفصل**
>
> - **الجزء:** 5 — بناء LLM من الصفر
> - **المستوى:** خبير
> - **زمن القراءة:** 25 دقيقة
> - **المتطلبات:** الفصل 22: حفظ نقاط التفتيش ومراقبة التدريب


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم المرحلة 1: SFT (Supervised Fine-Tuning)
- فهم Self-Instruct: توليد بيانات SFT اصطناعيًا
- فهم المرحلة 2: بناء نموذج المكافأة (Reward Model)


## 1. لماذا يهمّك هذا الموضوع؟

> 🟢 **طبقة الحدس**

بعد كل الجهد المبذول في الفصول 18-22 (بيانات نظيفة، Tokenizer مُحسَّن، حجم مثالي وفق Chinchilla، تدريب موزَّع مستقر)، ينتج لدينا ما يُسمَّى **النموذج الأساسي (Base Model)**. لكن هذا النموذج، رغم قدرته اللغوية القوية، له مشكلة جوهرية: **لم يُدرَّب على "اتباع التعليمات" أو "المحادثة"، بل فقط على "إكمال النص الأكثر احتمالاً إحصائيًا"**.

لو سألت نموذجًا أساسيًا خامًا: "ما هي عاصمة فرنسا؟"، فإن أكثر إكمال محتمل إحصائيًا للنص قد لا يكون "باريس" بل قد يكون سؤالاً آخر مشابهًا (لأن بيانات الويب تحتوي قوائم أسئلة متتالية أكثر من إجابات مباشرة)! هذا الفارق بين "نموذج يُكمل النص بإحصائية عالية" و"مساعد يفهم التعليمات ويستجيب بفائدة وأمان" هو بالضبط ما تُعالجه مرحلة **ما بعد التدريب (Post-Training)**.


## 2. الخريطة الذهنية


```text
خط إنتاج ما بعد التدريب الكامل
│
├── المرحلة 0: Base Model (نتاج الفصول 18-22)
│     نموذج "إكمال نص" خام، قوي لغويًا لكن غير مُطيع للتعليمات
│
├── المرحلة 1: SFT (Supervised Fine-Tuning)
│     تدريب على أزواج (تعليمة، استجابة مثالية) بإشراف مباشر
│     → ينتج "نموذج يتبع التعليمات" لكن دون تفضيل دقيق بين استجابات جيدة متعددة
│
├── المرحلة 2: بناء نموذج المكافأة (Reward Model - RM)
│     تدريب نموذج منفصل يتعلم "تقييم" جودة استجابة معينة
│     يُدرَّب على بيانات تفضيل بشرية (مقارنات ثنائية: استجابة A أفضل من B)
│
├── المرحلة 3: المحاذاة النهائية (Alignment)
│     RLHF (PPO) - الطريق الكلاسيكي الأصلي (معقد، يتطلب 4 نماذج في الذاكرة)
│     DPO/KTO/ORPO - البدائل الحديثة الأبسط (نتعلم مباشرة من التفضيلات
│                     دون الحاجة لنموذج مكافأة منفصل أو RL فعلي)
│
└── اعتبارات إضافية
      Self-Instruct (توليد بيانات SFT اصطناعيًا)
      Model Collapse (خطر التدريب على بيانات مولَّدة ذاتيًا بإفراط)
```


> 📌 **ملاحظة مهمة**: الجزء السابع من هذا الكتاب (PEFT/LoRA/QLoRA, RLHF, DPO/KTO/ORPO) يغطي هذه التقنيات **بعمق تقني كامل** مع كل الاشتقاقات الرياضية والأكواد التفصيلية. هذا الفصل (23) يُقدِّم **خريطة الطريق الشاملة والربط المنطقي** بين كل هذه المراحل كخط إنتاج متكامل واحد، تحضيرًا لرؤية الصورة الكاملة عمليًا في المشروع الختامي، بينما التفاصيل العميقة لكل تقنية (اشتقاق PPO، معادلة DPO الكاملة، إلخ) موجودة في الجزء السابع.


## 3. الشرح المتدرج

### 3.1 المرحلة 1: SFT (Supervised Fine-Tuning)

> 🟡 **طبقة الآلية**

الفكرة بسيطة مفاهيميًا: نأخذ النموذج الأساسي، ونُكمِل تدريبه (Fine-tune) على مجموعة بيانات مُنسَّقة خصيصًا كأزواج (تعليمة → استجابة مثالية مكتوبة/مُراجَعة بشريًا):


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset

class InstructionTuningDataset(Dataset):
    """
    مجموعة بيانات SFT: كل عينة هي محادثة مُنسَّقة بقالب موحَّد،
    حيث تُحسَب دالة الخسارة فقط على جزء "استجابة المساعد"
    (Response Masking)، وليس على نص التعليمة نفسها، لأن هدفنا
    تعليم النموذج "كيف يستجيب" وليس "كيف يكتب تعليمات".
    """
    def __init__(self, conversations: list, tokenizer, max_length: int = 2048):
        """
        conversations: قائمة من القواميس بالشكل:
        {"system": "...", "user": "...", "assistant": "..."}
        """
        self.conversations = conversations
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.conversations)

    def __getitem__(self, idx):
        conv = self.conversations[idx]

        # بناء القالب الكامل بالرموز الخاصة التي أضفناها عند تدريب
        # Tokenizer في الفصل 19 (<|system|>, <|user|>, <|assistant|>)
        system_part = f"<|system|>{conv.get('system', 'أنت مساعد ذكي ومفيد.')}"
        user_part = f"<|user|>{conv['user']}"
        assistant_part = f"<|assistant|>{conv['assistant']}<|endoftext|>"

        prompt_text = system_part + user_part + "<|assistant|>"
        full_text = system_part + user_part + assistant_part

        prompt_ids = self.tokenizer.encode(prompt_text).ids
        full_ids = self.tokenizer.encode(full_text).ids

        full_ids = full_ids[:self.max_length]
        labels = full_ids.copy()

        # Response Masking: نُخفي (بقيمة -100 التي يتجاهلها PyTorch تلقائيًا
        # في حساب CrossEntropyLoss) كل الأجزاء التي ليست من استجابة المساعد
        prompt_length = min(len(prompt_ids), len(labels))
        for i in range(prompt_length):
            labels[i] = -100

        return {
            "input_ids": torch.tensor(full_ids, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

def sft_training_step(model, batch, optimizer):
    """
    خطوة تدريب SFT واحدة. لاحظ أن الخسارة محسوبة فقط على الأجزاء
    غير المُخفية (استجابة المساعد)، بفضل قيمة -100 في labels التي
    يتجاهلها nn.CrossEntropyLoss تلقائيًا (ignore_index=-100 افتراضيًا).
    """
    optimizer.zero_grad()
    outputs = model(batch["input_ids"])
    logits = outputs[:, :-1, :].contiguous()
    labels = batch["labels"][:, 1:].contiguous()

    loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
    loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    return loss.item()


### 3.2 Self-Instruct: توليد بيانات SFT اصطناعيًا

> 🔴 **طبقة التعمّق**

جمع آلاف/عشرات آلاف أزواج (تعليمة، استجابة) عالية الجودة **بشريًا** مكلف جدًا من ناحية الوقت والمال. تقنية **Self-Instruct** (Wang et al., 2022) تُقدِّم حلاً: استخدام نموذج لغوي قوي موجود مسبقًا (مثل GPT-4) لتوليد تعليمات واستجابات اصطناعية جديدة، بدءًا من مجموعة صغيرة (Seed) من الأمثلة المكتوبة يدويًا.


In [ ]:
def self_instruct_pipeline_concept(seed_tasks: list,
                                     num_instructions_to_generate: int,
                                     teacher_model_generate_fn) -> list:
    """
    محاكاة مفاهيمية لخط إنتاج Self-Instruct لتوليد بيانات SFT اصطناعية.
    teacher_model_generate_fn: دالة تستدعي نموذجًا قويًا موجودًا (GPT-4 مثلاً)
    لتوليد نص من prompt معين.
    """
    generated_examples = []
    task_pool = seed_tasks.copy()

    while len(generated_examples) < num_instructions_to_generate:
        # 1. اختيار عينة عشوائية من المهام الموجودة (بذرة + مولَّدة سابقًا)
        #    كأمثلة توضيحية (Few-shot) لتوجيه النموذج المعلِّم
        import random
        few_shot_examples = random.sample(task_pool, min(4, len(task_pool)))

        # 2. بناء prompt يطلب من النموذج المعلِّم توليد تعليمة جديدة مختلفة
        generation_prompt = build_instruction_generation_prompt(few_shot_examples)
        new_instruction = teacher_model_generate_fn(generation_prompt)

        # 3. فلترة الجودة: التحقق من عدم التكرار الشديد مع المهام الموجودة
        if is_sufficiently_novel(new_instruction, task_pool):
            # 4. توليد الاستجابة المثالية لهذه التعليمة الجديدة بنفس النموذج المعلِّم
            response_prompt = f"أجب على التعليمة التالية بدقة ووضوح:\n{new_instruction}"
            new_response = teacher_model_generate_fn(response_prompt)

            new_example = {"instruction": new_instruction, "response": new_response}
            generated_examples.append(new_example)
            task_pool.append(new_example)  # إثراء البركة للتكرارات التالية

    return generated_examples

def build_instruction_generation_prompt(few_shot_examples: list) -> str:
    """بناء prompt يحتوي أمثلة توضيحية لتوجيه النموذج لتوليد تعليمة جديدة بنفس الأسلوب"""
    examples_text = "\n\n".join([
        f"تعليمة: {ex['instruction']}" for ex in few_shot_examples
    ])
    return (f"فيما يلي أمثلة على تعليمات متنوعة:\n\n{examples_text}\n\n"
            f"اكتب تعليمة جديدة مختلفة تمامًا عن الأمثلة أعلاه:\nتعليمة:")

def is_sufficiently_novel(new_instruction: str, existing_pool: list,
                            similarity_threshold: float = 0.7) -> bool:
    """فحص تشابه (مبسَّط هنا) للتأكد من تنوع البيانات المولَّدة"""
    # في التطبيق الفعلي: نستخدم ROUGE-L أو تشابه embeddings دلالي
    # هنا تبسيط مفاهيمي فقط
    return True  # افتراضيًا للتوضيح المفاهيمي


> ⚠️ **خطأ شائع وخطير: Model Collapse**. إذا دُرِّب نموذج بشكل متكرر عبر أجيال متعددة على بيانات **مولَّدة بالكامل من نماذج سابقة من نفس العائلة** (بدلاً من بيانات بشرية أصلية حقيقية)، فإن جودة النموذج تتدهور تدريجيًا عبر الأجيال — ظاهرة تُسمَّى **Model Collapse** (Shumailov et al., 2023، نُشرت في Nature). السبب: كل جيل يفقد تدريجيًا "الذيول النادرة" من التوزيع الحقيقي (الحالات النادرة لكن المهمة في اللغة الطبيعية)، ويتراكم الانحياز نحو "متوسط" ضيق وأقل تنوعًا. الحل العملي: **الحفاظ دائمًا على نسبة كافية من البيانات البشرية الأصلية** في أي مزيج تدريب يتضمن بيانات اصطناعية، ولا الاعتماد الكلي على بيانات مولَّدة ذاتيًا عبر أجيال متكررة.

### 3.3 المرحلة 2: بناء نموذج المكافأة (Reward Model)

نموذج المكافأة (RM) هو نموذج Transformer (غالبًا نفس معمارية النموذج الأساسي، لكن برأس إخراج (Head) مُعدَّل ليُنتج **قيمة عددية واحدة** بدلاً من توزيع احتمالي عبر المفردات). يُدرَّب على بيانات تفضيل بشرية: لكل تعليمة، يُقدَّم للمُقيِّم البشري زوج من الاستجابات (A وB)، ويختار أيهما أفضل.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class RewardModel(nn.Module):
    """
    نموذج المكافأة: يأخذ معمارية نموذج اللغة الأساسي (المُدرَّب مسبقًا
    وربما بعد SFT)، ويستبدل رأس الإخراج (الذي كان يُنتج توزيعًا على
    المفردات) برأس انحدار بسيط يُنتج قيمة عددية واحدة (Scalar Reward)
    تُمثِّل "جودة" الاستجابة الكاملة من منظور التفضيلات البشرية.
    """
    def __init__(self, base_transformer_model, hidden_size: int):
        super().__init__()
        self.transformer = base_transformer_model
        # رأس المكافأة: طبقة خطية بسيطة تُسقِط التمثيل المخفي لقيمة واحدة
        self.reward_head = nn.Linear(hidden_size, 1)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        # نأخذ التمثيل المخفي عند آخر رمز في التسلسل (يُلخِّص الاستجابة كاملة)
        hidden_states = self.transformer(input_ids, return_hidden_states=True)
        last_token_hidden = hidden_states[:, -1, :]  # [batch, hidden_size]
        reward_scalar = self.reward_head(last_token_hidden)  # [batch, 1]
        return reward_scalar.squeeze(-1)  # [batch]

def reward_model_training_step(reward_model: RewardModel,
                                 chosen_input_ids: torch.Tensor,
                                 rejected_input_ids: torch.Tensor,
                                 optimizer) -> float:
    """
    خطوة تدريب نموذج المكافأة باستخدام دالة خسارة Bradley-Terry
    (التصنيف الثنائي الاحتمالي، أساس كل نماذج المكافأة في RLHF):

    L = -log(σ(r_chosen - r_rejected))

    هذه الصيغة تدفع النموذج لإعطاء مكافأة أعلى للاستجابة المُفضَّلة بشريًا
    (chosen) مقارنة بالاستجابة المرفوضة (rejected)، دون الحاجة لقيم
    مكافأة "مطلقة" بل فقط لـ"ترتيب نسبي" صحيح بين الأزواج.
    """
    optimizer.zero_grad()

    reward_chosen = reward_model(chosen_input_ids)      # [batch]
    reward_rejected = reward_model(rejected_input_ids)  # [batch]

    # خسارة Bradley-Terry: نريد أن يكون σ(r_chosen - r_rejected) قريبًا من 1
    loss = -F.logsigmoid(reward_chosen - reward_rejected).mean()

    loss.backward()
    optimizer.step()

    # دقة الترتيب: كم نسبة الأزواج التي أعطاها النموذج مكافأة صحيحة الترتيب
    accuracy = (reward_chosen > reward_rejected).float().mean().item()

    return loss.item(), accuracy


### 3.4 المرحلة 3أ: RLHF الكلاسيكي بـ PPO (نظرة شاملة)


In [ ]:
"""
RLHF بـ PPO (Proximal Policy Optimization) يتطلب الاحتفاظ بأربعة نماذج
في الذاكرة في وقت واحد أثناء التدريب، وهذا يُفسِّر لماذا هذا المسار
مُكلِف حسابيًا جدًا مقارنة بالبدائل الحديثة (DPO وما شابه):

1. Policy Model (π_θ): النموذج الذي نُحسِّنه فعليًا (يبدأ كنسخة من نموذج SFT)
2. Reference Model (π_ref): نسخة مجمَّدة من نموذج SFT، تُستخدَم فقط لحساب
   عقوبة KL-Divergence لمنع π_θ من الانحراف بعيدًا جدًا عن سلوكه الأصلي
   (الانحراف المفرط قد يُنتج استجابات "تخدع" نموذج المكافأة لكنها غير متماسكة)
3. Reward Model (RM): النموذج المُدرَّب في القسم 3.3، مُجمَّد أثناء RLHF،
   يُعطي إشارة "المكافأة" لكل استجابة يولدها Policy Model
4. Value Model (Critic): نموذج يُقدِّر "القيمة المتوقعة" لحالة معينة،
   يُستخدَم لتقليل تباين تقدير الـ Advantage في خوارزمية PPO

معادلة الهدف الكلي التي يُحسِّنها PPO في سياق RLHF:

objective(θ) = E[r_φ(x,y) - β·KL(π_θ(y|x) || π_ref(y|x))]

حيث:
- r_φ(x,y): مكافأة الاستجابة y للمدخل x وفق نموذج المكافأة
- β: معامل يتحكم في قوة عقوبة الانحراف عن π_ref (قيمة نموذجية: 0.01-0.2)
- KL term: يمنع "اختراق" نموذج المكافأة (Reward Hacking) عبر استجابات
  غريبة تُحقق مكافأة عالية لكنها بعيدة عن اللغة الطبيعية المتماسكة
"""

def ppo_rlhf_training_loop_concept(policy_model, reference_model,
                                     reward_model, value_model,
                                     prompts_batch, optimizer, kl_coefficient=0.1):
    """
    هيكل مفاهيمي مبسَّط لحلقة تدريب PPO في سياق RLHF.
    التفاصيل الرياضية الكاملة (GAE، Clipping، إلخ) مُغطَّاة بعمق في الجزء السابع.
    """
    # 1. توليد استجابات من Policy Model الحالي لكل prompt في الدفعة
    generated_responses = policy_model.generate(prompts_batch)

    # 2. حساب المكافأة الخام من نموذج المكافأة المُجمَّد
    with torch.no_grad():
        raw_rewards = reward_model(generated_responses)

    # 3. حساب احتمالات log-probability من كل من Policy وReference
    policy_log_probs = policy_model.compute_log_probs(generated_responses)
    with torch.no_grad():
        reference_log_probs = reference_model.compute_log_probs(generated_responses)

    # 4. حساب عقوبة KL وطرحها من المكافأة الخام (منع الانحراف المفرط)
    kl_penalty = kl_coefficient * (policy_log_probs - reference_log_probs)
    adjusted_rewards = raw_rewards - kl_penalty

    # 5. حساب الـ Advantage باستخدام Value Model (GAE - تفصيل في الجزء السابع)
    with torch.no_grad():
        values = value_model(generated_responses)
    advantages = adjusted_rewards - values  # تبسيط؛ GAE الفعلي أكثر تعقيدًا

    # 6. تحديث Policy Model عبر هدف PPO المُقلَّم (Clipped Objective)
    # (التفاصيل الكاملة لـ clipping ratio في الجزء السابع)
    ppo_loss = compute_ppo_clipped_loss(policy_log_probs, advantages)

    optimizer.zero_grad()
    ppo_loss.backward()
    optimizer.step()

    return ppo_loss.item()


### 3.5 المرحلة 3ب: DPO — البديل الحديث الأبسط

> 💡 **طبقة الآلية المبسَّطة**

اكتشاف DPO (Rafailov et al., 2023) الأساسي: يمكن **رياضيًا** إعادة صياغة هدف RLHF بالكامل (بما فيه نموذج المكافأة الضمني) كدالة خسارة مباشرة على بيانات التفضيل، **دون الحاجة لتدريب نموذج مكافأة منفصل، ودون الحاجة لخوارزمية RL فعلية (PPO) على الإطلاق**!

$$\mathcal{L}_{DPO}(\theta) = -\mathbb{E}\left[\log \sigma\left(\beta \log\frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \beta \log\frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}\right)\right]$$

حيث $y_w$ هي الاستجابة المُفضَّلة (winning) و$y_l$ هي المرفوضة (losing). هذا يعني أن التدريب يصبح بسيطًا تمامًا مثل SFT: مجرد تمرير أمامي وخلفي عبر نموذجين (Policy وReference فقط، بدلاً من 4 نماذج في PPO)، بدون أي تعقيد RL.


In [ ]:
def dpo_loss_function(policy_chosen_logps: torch.Tensor,
                        policy_rejected_logps: torch.Tensor,
                        reference_chosen_logps: torch.Tensor,
                        reference_rejected_logps: torch.Tensor,
                        beta: float = 0.1) -> torch.Tensor:
    """
    دالة خسارة DPO الكاملة، كما اشتُقَّت في الورقة الأصلية.
    هذا هو "قلب" تبسيط RLHF إلى مجرد Fine-tuning مباشر بدون RL.
    (تفصيل الاشتقاق الكامل من هدف RLHF الأصلي موجود في الجزء السابع)
    """
    # نسبة لوغاريتمية الاحتمال (Policy مقابل Reference) لكل من الاستجابة
    # المُفضَّلة والمرفوضة
    policy_logratios = policy_chosen_logps - policy_rejected_logps
    reference_logratios = reference_chosen_logps - reference_rejected_logps

    logits = beta * (policy_logratios - reference_logratios)

    # نفس صيغة Bradley-Terry المستخدمة في تدريب نموذج المكافأة تمامًا،
    # لكن مُطبَّقة مباشرة على النموذج النهائي دون وسيط (نموذج مكافأة منفصل)
    loss = -F.logsigmoid(logits).mean()

    # مقاييس إضافية مفيدة للمراقبة أثناء التدريب
    chosen_rewards = beta * (policy_chosen_logps - reference_chosen_logps).detach()
    rejected_rewards = beta * (policy_rejected_logps - reference_rejected_logps).detach()
    reward_accuracy = (chosen_rewards > rejected_rewards).float().mean()

    return loss, reward_accuracy, chosen_rewards.mean(), rejected_rewards.mean()


### 3.6 جدول مقارنة شامل: RLHF/PPO مقابل DPO ومشتقاتها


In [ ]:
def compare_alignment_methods():
    methods = {
        "RLHF (PPO)": {
            "عدد النماذج في الذاكرة": 4,
            "يحتاج Reward Model منفصل": "نعم",
            "استقرار التدريب": "صعب (حساس جدًا للمعاملات)",
            "التعقيد الهندسي": "عالٍ جدًا",
        },
        "DPO": {
            "عدد النماذج في الذاكرة": 2,
            "يحتاج Reward Model منفصل": "لا",
            "استقرار التدريب": "أكثر استقرارًا بكثير",
            "التعقيد الهندسي": "بسيط (شبيه بـ SFT)",
        },
        "KTO": {
            "عدد النماذج في الذاكرة": 2,
            "يحتاج Reward Model منفصل": "لا",
            "استقرار التدريب": "جيد، يعمل حتى ببيانات غير مزدوجة (binary فقط)",
            "التعقيد الهندسي": "بسيط",
        },
        "ORPO": {
            "عدد النماذج في الذاكرة": 1,
            "يحتاج Reward Model منفصل": "لا، ولا حتى Reference Model!",
            "استقرار التدريب": "جيد، يدمج SFT والمحاذاة في خطوة واحدة",
            "التعقيد الهندسي": "الأبسط على الإطلاق",
        },
    }
    for method, details in methods.items():
        print(f"\n{method}:")
        for k, v in details.items():
            print(f"   {k}: {v}")

compare_alignment_methods()


> 📌 تفصيل كامل لاشتقاق كل من PPO وDPO وKTO وORPO، مع أكواد تدريب كاملة قابلة للتشغيل، موجود في **الجزء السابع** من هذا الكتاب. الهدف هنا هو فهم **موقع كل تقنية ضمن خط الإنتاج الكلي**.


## 4. مثال محلول خطوة بخطوة

**سيناريو متكامل**: لديك نموذج أساسي (Base) بحجم 150 مليون معامل (نتاج الفصول 18-22). تريد تحويله لمساعد محادثة. تتبع الخطوات التالية:


In [ ]:
def complete_post_training_pipeline_walkthrough():
    """
    محاكاة مفاهيمية لخط الإنتاج الكامل من Base إلى نموذج محاذى نهائيًا،
    موضحة تدرج كل مرحلة وأثرها المتوقع على سلوك النموذج.
    """
    stages = [
        {
            "stage": "0. Base Model",
            "data": "15 مليار رمز (ويب منظَّف، ويكيبيديا، كتب)",
            "behavior": "يُكمل النص إحصائيًا، لا يفهم 'التعليمات' كمفهوم",
            "example_prompt": "ما هي عاصمة فرنسا؟",
            "example_output": "ما هي عاصمة ألمانيا؟ ما هي عاصمة إيطاليا؟ ...",
        },
        {
            "stage": "1. بعد SFT",
            "data": "~50,000 زوج (تعليمة، استجابة) عالية الجودة",
            "behavior": "يفهم بنية 'سؤال→إجابة مباشرة'، يتبع تعليمات بسيطة",
            "example_prompt": "ما هي عاصمة فرنسا؟",
            "example_output": "عاصمة فرنسا هي باريس.",
        },
        {
            "stage": "2. بعد RM (لا يُغيِّر سلوك التوليد مباشرة)",
            "data": "~10,000 زوج تفضيل بشري (مُفضَّل مقابل مرفوض)",
            "behavior": "النموذج الأصلي لم يتغير بعد، فقط بُنيَ 'حَكَم' منفصل",
            "example_prompt": "(غير مطبَّق في هذه المرحلة مباشرة)",
            "example_output": "RM يُعطي: الإجابة المباشرة = مكافأة عالية، الإجابة المطوَّلة بلا داعٍ = مكافأة أقل",
        },
        {
            "stage": "3. بعد DPO/RLHF",
            "data": "نفس بيانات التفضيل، لكن تُستخدَم لتحديث النموذج مباشرة",
            "behavior": "استجابات أكثر دقة، أكثر أمانًا، أقرب لتفضيلات المُقيِّمين البشريين",
            "example_prompt": "اشرح لي كيف أصنع متفجرات في المنزل",
            "example_output": "لا يمكنني المساعدة في هذا الطلب لأنه قد يُسبب ضررًا جسيمًا...",
        },
    ]

    for s in stages:
        print(f"\n{'='*70}")
        print(f"المرحلة: {s['stage']}")
        print(f"البيانات المُستخدَمة: {s['data']}")
        print(f"السلوك الناتج: {s['behavior']}")
        print(f"مثال - المدخل: {s['example_prompt']}")
        print(f"مثال - المخرج المتوقع: {s['example_output']}")

complete_post_training_pipeline_walkthrough()


لاحظ التدرج الواضح: Base → SFT (يتعلم "شكل" الاستجابة المفيدة) → RM (يبني "حَكَمًا" منفصلاً) → DPO/RLHF (يُحسِّن النموذج فعليًا وفق أحكام ذلك الحَكَم، بما فيها رفض الطلبات الضارة).


## 5. نصائح وحيل

- **جودة بيانات SFT أهم من كميتها**: عشرة آلاف مثال عالي الجودة ومُراجَع بعناية غالبًا يتفوق على مئة ألف مثال متوسط الجودة (كما أظهرت ورقة LIMA الشهيرة: "Less Is More for Alignment").
- ابدأ بـ **DPO بدلاً من RLHF/PPO** كخيار افتراضي لمعظم المشاريع، إلا إذا كان لديك بنية تحتية ناضجة وخبرة عميقة بـ RL، لأن DPO أبسط وأكثر استقرارًا بشكل كبير بنتائج مماثلة في معظم الحالات العملية.
- حافظ دائمًا على **نسخة من Reference Model** (عادة نموذج SFT قبل أي محاذاة) لحساب عقوبة KL، حتى في DPO — هذا يمنع النموذج من "نسيان" قدراته اللغوية الأساسية أثناء التحسين المفرط لتفضيلات ضيقة.
- راقب "Reward Hacking" بعناية: أحيانًا يجد النموذج "ثغرات" تُحقق مكافأة عالية (مثل الإطالة غير المبررة للإجابات لأن المُقيِّمين البشريين كانوا يفضلون إجابات أطول قليلاً في المتوسط) دون تحسين حقيقي في الجودة.


## 6. أخطاء شائعة ومصائد

| الخطأ | النتيجة | الحل |
|---|---|---|
| تخطي مرحلة SFT والانتقال مباشرة لـ RLHF/DPO على النموذج الأساسي | عدم استقرار شديد (Base Model لا "يفهم" مفهوم المحادثة أصلاً) | SFT إلزامية دائمًا كخطوة أولى |
| الاعتماد الكلي على بيانات اصطناعية (Self-Instruct) دون أي بيانات بشرية | خطر Model Collapse التدريجي عبر الأجيال | الحفاظ على نسبة كافية من البيانات البشرية الأصلية دائمًا |
| إهمال عقوبة KL في RLHF/DPO | النموذج "يختراق" نموذج المكافأة بإجابات غريبة غير متماسكة لغويًا | ضبط معامل β بعناية، ومراقبة تماسك الاستجابات باستمرار |
| الاكتفاء بمقياس Reward Model فقط لتقييم جودة النموذج النهائي | قد يُخفي مشاكل حقيقية (Reward Hacking) غير مرئية لنموذج المكافأة نفسه | دمج تقييم بشري دوري + مقاييس تلقائية متعددة (الجزء التاسع) |


## 7. جدول مقارنة

| المرحلة | المدخل | المخرج | نوع التعلم | الهدف |
|---|---|---|---|---|
| Pre-training (الفصول 18-22) | نص خام ضخم | Base Model | تعلم ذاتي-خاضع للإشراف (Self-Supervised) | قدرة لغوية ومعرفية عامة |
| SFT | أزواج (تعليمة، استجابة) | نموذج يتبع التعليمات | تعلم خاضع للإشراف المباشر | تعلّم "شكل" الاستجابة المفيدة |
| Reward Modeling | أزواج تفضيل (مُفضَّل، مرفوض) | Reward Model منفصل | تصنيف ثنائي (Bradley-Terry) | بناء "حَكَم" آلي لجودة الاستجابات |
| RLHF/DPO | نفس بيانات التفضيل أو عينات جديدة | نموذج مُحاذى نهائيًا | RL (PPO) أو تحسين مباشر (DPO) | محاذاة السلوك مع التفضيلات البشرية |


## 8. ملخص أفضل الممارسات

- اتبع التسلسل الصحيح دائمًا: Pre-training → SFT → (RM) → RLHF/DPO، ولا تتخطَّ أي مرحلة.
- فضِّل DPO كخيار افتراضي أبسط وأكثر استقرارًا، وانتقل لـ PPO/RLHF فقط عند الحاجة لمرونة إضافية مُبرَّرة.
- حافظ على جودة بيانات عالية في كل مرحلة، خصوصًا SFT — الجودة هنا أهم من الكمية بشكل حاسم.
- راقب نسبة البيانات الاصطناعية إلى البشرية بعناية لتفادي Model Collapse.
- استخدم عقوبة KL في كل تقنيات المحاذاة لمنع الانحراف المفرط عن السلوك اللغوي الطبيعي السليم.
- قيِّم النموذج النهائي بمزيج من مقاييس تلقائية وتقييم بشري، وليس بمقياس واحد فقط.


## 9. أسئلة مقابلات

**سؤال 1**: لماذا لا يمكن تطبيق RLHF أو DPO مباشرة على النموذج الأساسي (Base Model) دون المرور بمرحلة SFT أولاً؟

> **الإجابة النموذجية**: النموذج الأساسي مُدرَّب فقط على هدف "إكمال النص الأكثر احتمالاً إحصائيًا"، وليس لديه أي مفهوم مُتعلَّم عن "بنية المحادثة" (تعليمة من مستخدم تليها استجابة من مساعد). بيانات التفضيل المستخدمة في RLHF/DPO (أزواج استجابات مُفضَّلة/مرفوضة لتعليمة معينة) تفترض ضمنًا أن النموذج **يستطيع أصلاً توليد استجابات متماسكة بصيغة محادثة** لمقارنة جودتها. تطبيق هذه التقنيات على نموذج لم يتعلم بعد "شكل" الاستجابة المناسب سينتج تدريبًا غير مستقر تمامًا، لأن الفضاء الذي يُستكشف (استجابات النموذج الأساسي العشوائية نسبيًا) بعيد جدًا عن الفضاء الذي صُمِّمت بيانات التفضيل لتقييمه. لذلك SFT ضرورية أولاً لتأسيس "أرضية" سلوكية معقولة (نموذج يتبع التعليمات بأسلوب محادثة) قبل البدء بالتحسين الدقيق وفق التفضيلات.

**سؤال 2**: ما السبب الرياضي الجوهري الذي يسمح لـ DPO بتحقيق نفس هدف RLHF دون الحاجة لنموذج مكافأة منفصل أو خوارزمية RL فعلية؟

> **الإجابة النموذجية**: في RLHF، الحل الأمثل النظري لمسألة التحسين $\max_\pi \mathbb{E}[r(x,y)] - \beta \cdot KL(\pi||\pi_{ref})$ له صيغة مغلقة (Closed-form) تربط مباشرة بين دالة المكافأة المثلى والسياسة المثلى الناتجة عنها: $r(x,y) = \beta \log\frac{\pi^*(y|x)}{\pi_{ref}(y|x)} + \text{const}$. اكتشاف DPO هو أنه بدلاً من تعلّم $r(x,y)$ أولاً (عبر نموذج مكافأة منفصل) ثم البحث عن $\pi^*$ التي تُعظِّمه (عبر RL)، يمكن **تعويض** هذه العلاقة مباشرة في دالة خسارة Bradley-Terry الأصلية المستخدمة لتدريب نموذج المكافأة، مما يُنتج دالة خسارة تعتمد فقط على $\pi_\theta$ و $\pi_{ref}$ مباشرة — وتُحسَّن بالتدرج الانحداري العادي (Gradient Descent) تمامًا مثل أي Fine-tuning عادي، دون الحاجة لأي خطوة RL وسيطة أو نموذج مكافأة منفصل على الإطلاق.

**سؤال 3**: ما هي ظاهرة Model Collapse، ولماذا تُعد خطرًا متزايد الأهمية مع ازدياد الاعتماد على تقنيات مثل Self-Instruct؟

> **الإجابة النموذجية**: Model Collapse ظاهرة تحدث عندما تُدرَّب نماذج لغوية بشكل متكرر عبر أجيال متتالية بشكل كبير على بيانات مولَّدة بواسطة نماذج سابقة من نفس العائلة (بدلاً من بيانات بشرية أصلية)، مما يُسبب تدهورًا تدريجيًا في جودة وتنوع مخرجات النموذج عبر الأجيال. السبب الرياضي: كل جيل من التوليد يميل لإنتاج عينات أقرب لـ"مركز" توزيع البيانات (الأنماط الأكثر شيوعًا) ويفقد تدريجيًا "الذيول" النادرة من التوزيع الحقيقي (المعلومات النادرة لكن الصحيحة والمفيدة)، بسبب تراكم أخطاء التقريب الإحصائي عبر كل جيل من إعادة التوليد. مع تزايد استخدام تقنيات مثل Self-Instruct لتوليد بيانات تدريب اصطناعيًا بكميات كبيرة (لتقليل التكلفة مقارنة بجمع بيانات بشرية)، ومع تزايد نسبة المحتوى المولَّد بالذكاء الاصطناعي على الإنترنت نفسه (الذي يُستخدَم لاحقًا كبيانات تدريب لنماذج مستقبلية)، يُصبح خطر تلوث سلسلة التدريب بهذه الظاهرة تهديدًا حقيقيًا طويل الأمد يتطلب حرصًا متعمَّدًا على الحفاظ على مصادر بيانات بشرية أصلية موثوقة ومتنوعة.


## 10. تمرين عملي

باستخدام الأكواد المفاهيمية أعلاه (`InstructionTuningDataset`، `RewardModel`، `dpo_loss_function`)، صمِّم خط إنتاج كامل (حتى لو على نموذج صغير جدًا توضيحي وبيانات اصطناعية محدودة) يُنفِّذ:
1. SFT على 50 مثال محادثة اصطناعي.
2. بناء Reward Model مُبسَّط على 20 زوج تفضيل.
3. تطبيق خطوة DPO واحدة وقياس `reward_accuracy` قبل وبعد.

ناقش في تقرير قصير: هل لاحظت تحسنًا في `reward_accuracy`؟ ما التحديات العملية التي واجهتها حتى على هذا النطاق المصغَّر؟


## 11. مراجع للتعمّق

- Ouyang et al. (2022), *"Training language models to follow instructions with human feedback"* (ورقة InstructGPT، أساس RLHF العملي).
- Rafailov et al. (2023), *"Direct Preference Optimization: Your Language Model is Secretly a Reward Model"* (ورقة DPO الأصلية).
- Wang et al. (2022), *"Self-Instruct: Aligning Language Models with Self-Generated Instructions"*.
- Zhou et al. (2023), *"LIMA: Less Is More for Alignment"*.
- Shumailov et al. (2024), *"AI models collapse when trained on recursively generated data"* (نُشرت في Nature).
- Ethayarajh et al. (2024), *"KTO: Model Alignment as Prospect Theoretic Optimization"*.
- Hong et al. (2024), *"ORPO: Monolithic Preference Optimization without Reference Model"*.


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — بناء LLM من الصفر / الفصل 23: خط إنتاج ما بعد التدريب.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
